# ENTRENAMIENTO DE LA CNN (Convolutional Neural Network)

### Importamos las librerías necesarias y el dataset de entrenamiento

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, MaxPooling2D, Activation, BatchNormalization
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
fashion_mnist_data = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist_data.load_data()


## Exploracion del Dataset

El dataset **Fashion MNIST** contiene 60000 imagenes de entrenamiento y 10000 de prueba, cada una de 28x28 pixeles en escala de grises. Las imagenes pertenecen a 10 clases distintas de prendas de vestir: Remera, Pantalon, Sueter, Vestido, Abrigo, Sandalia, Camisa, Zapatilla, Bolso y Bota.

In [ ]:
train_images.shape


In [ ]:
np.unique(train_labels)


In [ ]:
labels = [
    'Remera',
    'Pantalon',
    'Sueter',
    'Vestido',
    'Abrigo',
    'Sandalia',
    'Camisa',
    'Zapatilla',
    'Bolso',
    'Bota'
]


## Preprocesamiento

Normalizamos los valores de los pixeles al rango [0, 1] dividiendo por 255.0. Esto acelera la convergencia del gradiente descendente y evita que los valores grandes dominen el entrenamiento.

In [ ]:
train_images = train_images/255.0
test_images = test_images/255.0


## División Train / Validation

Separamos el conjunto de entrenamiento en **51000 muestras para entrenar (85%)** y **9000 para validacion (15%)**. El conjunto de validacion permite monitorear el rendimiento del modelo durante el entrenamiento y detectar overfitting.

In [ ]:
len(train_images) * .85 #51000
len(train_images) - 51000 #9000

X_train = train_images[:51000]
X_val= train_images[51000:]

Y_train = train_labels[:51000]
Y_val = train_labels[51000:]


## Visualización de Muestras

Inspeccionamos visualmente algunas imagenes del dataset junto con su etiqueta correspondiente para verificar la calidad de los datos y familiarizarnos con las clases.

In [ ]:
i = 100
img = train_images[i,:,:]
plt.imshow(img)
plt.show
print(f"label: {labels[train_labels[i]]}")


## Arquitectura de la CNN

Definimos una red convolucional secuencial con la siguiente estructura:

- **Bloque 1:** 2 capas Conv2D de 32 filtros (3x3, padding same) + ReLU + BatchNormalization + MaxPooling 2x2 + Dropout 0.25
- **Bloque 2:** 2 capas Conv2D de 64 filtros (3x3, padding same) + ReLU + BatchNormalization + MaxPooling 2x2 + Dropout 0.25
- **Clasificador:** Flatten, Dense 512 + ReLU + BatchNormalization + Dropout 0.5, Dense 10 + Softmax

Se eligieron filtros 3x3 por su eficiencia computacional, BatchNormalization para estabilizar el entrenamiento, MaxPooling para reducir dimensionalidad espacial y Dropout para prevenir overfitting.

In [ ]:
model = Sequential()
model.add(Conv2D(32, (3, 3), padding="same",input_shape=(28,28,1)))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=3))
model.add(Conv2D(32, (3, 3), padding="same"))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=3))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, (3, 3), padding="same"))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=3))
model.add(Conv2D(64, (3, 3), padding="same"))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=3))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512))
model.add(Activation("relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(10))
model.add(Activation("softmax"))


In [ ]:
train_images[..., np.newaxis].shape


## Compilacion y Entrenamiento

Compilamos el modelo con:
- **Optimizador:** Adam (tasa de adaptacion automatica)
- **Perdida:** sparse_categorical_crossentropy (ideal para etiquetas enteras)
- **Metrica:** accuracy

Entrenamos durante **80 epocas** con batches de **128 muestras**, utilizando el conjunto de validacion para supervisar el progreso y detectar overfitting.

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
history = model.fit(X_train[..., np.newaxis], Y_train, epochs=80, batch_size=128, validation_data=(X_val[..., np.newaxis], Y_val))


## Analisis del Historial de Entrenamiento

Visualizamos la evolucion de la perdida (loss) y la precision (accuracy) a lo largo de las epocas tanto para entrenamiento como para validacion. La convergencia de ambas curvas indica un buen ajuste; una divergencia creciente sugeriria overfitting.

In [ ]:
df = pd.DataFrame(history.history)
df


In [ ]:
loss_plot = df.plot(y='loss', title='loss vs epochs', legend=False)
loss_plot.set(xlabel='Epochs', ylabel='loss')


In [ ]:
acc_plot = df.plot(y='accuracy', title='accuracy vs epochs', legend=False)
acc_plot.set(xlabel='Epochs', ylabel='Accuracy')


## Evaluacion en el Conjunto de Test

Evaluamos el modelo final con los **10000 datos de test** no vistos durante el entrenamiento. Calculamos:

- **Perdida y precision global** en test
- **Classification Report:** precision, recall y f1-score por cada clase
- **Matriz de Confusion:** para identificar que pares de clases se confunden con mayor frecuencia (ej. Camisa vs Remera)

In [ ]:
print(test_images.shape)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(test_images[..., np.newaxis])
y_pred_classes = np.argmax(y_pred, axis=1)

test_loss, test_acc = model.evaluate(test_images[..., np.newaxis], test_labels, verbose=0)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

print("\nClassification Report:\n")
print(classification_report(test_labels, y_pred_classes))

cm = confusion_matrix(test_labels, y_pred_classes)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="coolwarm"
)

plt.xlabel("Prediccion")
plt.ylabel("Real")
plt.title("Matriz de Confusion")
plt.show()


## Prediccion sobre una Muestra Aleatoria

Seleccionamos una imagen aleatoria del conjunto de test y mostramos tanto la etiqueta real como la prediccion del modelo para verificar cualitativamente su desempeño.

In [ ]:
inx = np.random.choice(test_images.shape[0])
test_image = test_images[inx]
plt.imshow(test_image)
plt.show()
print(f'Label: {labels[test_labels[inx]]}')


In [ ]:
predictions = model.predict(test_image[np.newaxis, ..., np.newaxis])
print(f'Prediction: {labels[test_labels[inx]]}')


## Extraccion y Empaquetado de Pesos

Una vez entrenada la CNN, extraemos todos los parametros aprendidos (pesos, biases, medias y varianzas de BatchNormalization) y los serializamos en un archivo **.pkl**. Este archivo sera consumido por el motor de inferencia CUDA para ejecutar la red en GPU.

El proceso incluye:
1. **Extraccion** de pesos de cada capa (Conv2D, BatchNormalization, Dense)
2. **Validacion** de shapes segun la arquitectura esperada
3. **Exportacion** a `model_params.pkl`

In [ ]:
import pickle
import numpy as np

def extract_model_params(model) -> dict:
    """
    Extrae todos los parametros del modelo Keras y los exporta
    en el formato esperado por CUDAInferenceEngine.
    """
    params = {}

    LAYER_MAP = {
        'conv2d':   'conv2d',
        'conv2d_1': 'conv2d_1',
        'conv2d_2': 'conv2d_2',
        'conv2d_3': 'conv2d_3',
        'batch_normalization':   'batch_normalization',
        'batch_normalization_1': 'batch_normalization_1',
        'batch_normalization_2': 'batch_normalization_2',
        'batch_normalization_3': 'batch_normalization_3',
        'batch_normalization_4': 'batch_normalization_4',
        'dense':   'dense',
        'dense_1': 'dense_1',
    }

    for keras_name, cuda_key in LAYER_MAP.items():
        try:
            layer = model.get_layer(keras_name)
        except ValueError:
            print(f"  [WARN] Capa '{keras_name}' no encontrada en el modelo, se omite.")
            continue

        layer_type = layer.__class__.__name__

        if layer_type == 'Conv2D':
            weights, bias = layer.get_weights()
            params[cuda_key] = {
                'weights': weights.astype(np.float32),
                'bias':    bias.astype(np.float32),
            }
            print(f"  ok {keras_name:25s} -> weights{weights.shape}, bias{bias.shape}")

        elif layer_type == 'BatchNormalization':
            gamma, beta, moving_mean, moving_var = layer.get_weights()
            params[cuda_key] = {
                'gamma':       gamma.astype(np.float32),
                'beta':        beta.astype(np.float32),
                'moving_mean': moving_mean.astype(np.float32),
                'moving_var':  moving_var.astype(np.float32),
            }
            print(f"  ok {keras_name:25s} -> gamma{gamma.shape}, mean{moving_mean.shape}")

        elif layer_type == 'Dense':
            weights, bias = layer.get_weights()
            params[cuda_key] = {
                'weights': weights.astype(np.float32),
                'bias':    bias.astype(np.float32),
            }
            print(f"  ok {keras_name:25s} -> weights{weights.shape}, bias{bias.shape}")

        else:
            print(f"  -  {keras_name:25s} ({layer_type}) sin parametros, omitida.")

    return params


def validate_params(params: dict):
    print("\n--- Validacion de shapes ---")
    errors = []

    checks = {
        'conv2d':            ('Conv bloque1-1', ('weights', (3,3,1,32)),  ('bias', (32,))),
        'conv2d_1':          ('Conv bloque1-2', ('weights', (3,3,32,32)), ('bias', (32,))),
        'conv2d_2':          ('Conv bloque2-1', ('weights', (3,3,32,64)), ('bias', (64,))),
        'conv2d_3':          ('Conv bloque2-2', ('weights', (3,3,64,64)), ('bias', (64,))),
        'batch_normalization':   ('BN1', ('gamma', (32,)), ('moving_var', (32,))),
        'batch_normalization_1': ('BN2', ('gamma', (32,)), ('moving_var', (32,))),
        'batch_normalization_2': ('BN3', ('gamma', (64,)), ('moving_var', (64,))),
        'batch_normalization_3': ('BN4', ('gamma', (64,)), ('moving_var', (64,))),
        'batch_normalization_4': ('BN5', ('gamma', (512,)), ('moving_var', (512,))),
        'dense':             ('Dense-512', ('weights', (3136, 512)), ('bias', (512,))),
        'dense_1':           ('Dense-10',  ('weights', (512, 10)),   ('bias', (10,))),
    }

    for key, (desc, *field_checks) in checks.items():
        if key not in params:
            errors.append(f"  [ERROR] '{key}' ({desc}) ausente en params")
            continue
        for field, expected_shape in field_checks:
            actual = params[key][field].shape
            if actual != expected_shape:
                errors.append(f"  [ERROR] {key}.{field}: esperado {expected_shape}, obtenido {actual}")
            else:
                print(f"  ok {key}.{field}: {actual}")

    if errors:
        print("\nErrores encontrados:")
        for e in errors:
            print(e)
        raise ValueError("Validacion fallida.")
    else:
        print("\n  ok Todas las shapes son correctas.")


def export_params(params: dict, output_path: str = 'model_params.pkl'):
    with open(output_path, 'wb') as f:
        pickle.dump(params, f, protocol=pickle.HIGHEST_PROTOCOL)

    size_mb = sum(
        arr.nbytes
        for layer in params.values()
        for arr in layer.values()
    ) / (1024 ** 2)

    print(f"\n  ok Exportado en '{output_path}'")
    print(f"  ok Capas exportadas : {len(params)}")
    print(f"  ok Tamano total     : {size_mb:.2f} MB")

print("--- Extrayendo parametros del modelo ---")
params = extract_model_params(model)
validate_params(params)
export_params(params, output_path='model_params.pkl')


# Pipeline de Inferencia CUDA con los Pesos de la CNN

A continuacion implementamos el pipeline de inferencia completo en **CUDA C++**, compilado y ejecutado desde Python mediante **PyCUDA**. Los kernels implementados replican cada operacion de la red:
- Normalizacion de entrada
- Convolucion 2D (con padding y stride)
- ReLU (activacion)
- BatchNormalization
- MaxPooling 2x2
- Flatten
- Dense (matmul + bias)
- Softmax (numericamente estable)

Primero instalamos y configuramos PyCUDA.

In [ ]:
!pip install pycuda
!>/var/colab/app.log


In [ ]:
import pycuda.driver as cuda
import pycuda.autoinit
import pycuda.compiler
import warnings
warnings.filterwarnings('ignore')

print("PyCUDA importado exitosamente")
print(f"CUDA disponible: {cuda.Device.count()} dispositivo(s) detectado(s)")


## Definicion y Compilacion de Kernels CUDA

Se definen los kernels CUDA para cada operacion de la red y se compilan con **nvcc** en un fatbin compatible con multiples arquitecturas GPU (Pascal, Turing, Ampere, Ada Lovelace). Los kernels se organizan en un solo archivo fuente y se compilan con soporte para las generaciones sm_61 hasta sm_89.

In [ ]:
# ==================== KERNELS CUDA COMPILADOS ====================
CUDA_KERNELS_SOURCE = """
#include <float.h>
extern "C" {
// ============================================================
// normalize_kernel: Limita cada valor al rango [0, 1] mediante clamping.
// Recibe un vector de float32 y devuelve cada elemento recortado entre 0 y 1.
// Utilizado para asegurar que las activaciones no excedan el rango esperado.
// ============================================================
  __global__ void normalize_kernel(
      float *input, float *output, int total_elements
  ) {
      int idx = blockIdx.x * blockDim.x + threadIdx.x;
      if (idx < total_elements) {
          output[idx] = fmin(1.0f, fmax(0.0f, input[idx]));
      }
        }
// ============================================================
// preprocess_kernel: Transforma una imagen cruda desde bytes a float32 normalizado.
// Recibe una imagen en formato uint8 con 1, 3 o 4 canales.
// Si tiene múltiples canales, combina RGB mediante una suma ponderada que respeta
// la percepción de brillo del ojo humano (luminancia perceptual).
// Si es un solo canal, lo usa directamente.
// Luego divide por 255 para llevar los valores al rango [0, 1] e invierte los colores
// para que el fondo sea negro y el objeto blanco, coincidiendo con el formato
// con el que fue entrenado el modelo Fashion MNIST.
// Es el primer paso del pipeline de inferencia: sin él la red recibiría datos
// en un formato incompatible con el que aprendió durante el entrenamiento.
// ============================================================
  __global__ void preprocess_kernel(
      const unsigned char* input, float* output,
      int width, int height, int channels
  ) {
      int x = blockIdx.x * blockDim.x + threadIdx.x;
      int y = blockIdx.y * blockDim.y + threadIdx.y;
      if (x >= width || y >= height) return;

      int idx = y * width + x;
      float pixel;
      if (channels >= 3) {
          pixel = 0.299f * input[idx * channels + 0] +
                  0.587f * input[idx * channels + 1] +
                  0.114f * input[idx * channels + 2];
      } else {
          pixel = (float)input[idx];
      }
      output[idx] = 1.0f - (pixel / 255.0f);
  }
// ============================================================
// relu_kernel: Aplica la función de activación ReLU a cada elemento.
// Recibe un vector de float32 y devuelve max(0, valor) para cada posición.
// Introduce no-linealidad en la red, permitiendo que aprenda relaciones complejas.
// Sin activaciones no-lineales, la red sería equivalente a una composición de
// transformaciones lineales y no podría modelar funciones no triviales.
// ============================================================
  __global__ void relu_kernel(
      float *input, float *output, int total_elements
  ) {
      int idx = blockIdx.x * blockDim.x + threadIdx.x;
      if (idx < total_elements) {
          output[idx] = fmax(0.0f, input[idx]);
      }
  }
// ============================================================
// conv2d_kernel: Convolución 2D multicanal con sesgo.
// Recibe un tensor de entrada, un kernel de pesos y un vector de sesgos.
// Para cada posición espacial de salida y cada canal de salida, calcula la suma
// ponderada entre una ventana de la entrada y el kernel correspondiente en esa
// región, más el sesgo del canal. Cada kernel aprende a detectar un patrón visual
// específico (bordes, texturas, formas) en una vecindad local.
// Es la operación fundamental de extracción de características en la red.
// ============================================================
  __global__ void conv2d_kernel(
      const float *input, const float *weights, const float *bias,
      float *output,
      int batch_size, int height, int width, int in_channels,
      int out_channels, int kernel_size, int stride, int padding
  ) {
      int out_x = blockIdx.x * blockDim.x + threadIdx.x;
      int out_y = blockIdx.y * blockDim.y + threadIdx.y;
      int out_c = blockIdx.z;
      int batch = 0;
      if (out_x >= (width + 2*padding - kernel_size)/stride + 1) return;
      if (out_y >= (height + 2*padding - kernel_size)/stride + 1) return;
      if (out_c >= out_channels) return;
      float sum = bias[out_c];
      for (int kh = 0; kh < kernel_size; kh++) {
          for (int kw = 0; kw < kernel_size; kw++) {
              int in_y = out_y * stride + kh - padding;
              int in_x = out_x * stride + kw - padding;
              if (in_y >= 0 && in_y < height && in_x >= 0 && in_x < width) {
                  for (int ic = 0; ic < in_channels; ic++) {
                      int input_idx = ((in_y * width + in_x) * in_channels + ic);
                      int weight_idx = ((kh * kernel_size + kw) * in_channels + ic) * out_channels + out_c;
                      sum += input[input_idx] * weights[weight_idx];
                  }
              }
          }
      }
      int out_w = (width + 2*padding - kernel_size)/stride + 1;
      int out_idx = (out_y * out_w + out_x) * out_channels + out_c;
      output[out_idx] = sum;
  }
// ============================================================
// maxpool2d_kernel: Reduce la resolución espacial tomando el máximo en ventanas 2×2.
// Recibe un tensor tridimensional y devuelve otro con la mitad de alto y ancho.
// Para cada canal y cada bloque de 2×2 sin solapamiento, conserva el valor máximo.
// Esto disminuye la cantidad de parámetros en las capas siguientes, controla el
// sobreajuste y otorga cierta tolerancia a pequeñas traslaciones de la entrada.
// ============================================================
  __global__ void maxpool2d_kernel(
      const float *input, float *output,
      int height, int width, int channels
  ) {
      int out_x = blockIdx.x * blockDim.x + threadIdx.x;
      int out_y = blockIdx.y * blockDim.y + threadIdx.y;
      int c = blockIdx.z;
      int out_height = height / 2;
      int out_width = width / 2;
      if (out_x >= out_width || out_y >= out_height || c >= channels) return;
      int in_x = out_x * 2;
      int in_y = out_y * 2;
      float max_val = -FLT_MAX;
      for (int dy = 0; dy < 2; dy++) {
          for (int dx = 0; dx < 2; dx++) {
              int idx = ((in_y + dy) * width + (in_x + dx)) * channels + c;
              max_val = fmax(max_val, input[idx]);
          }
      }
      int out_idx = (out_y * out_width + out_x) * channels + c;
      output[out_idx] = max_val;
  }
// ============================================================
// matmul_kernel: Producto entre dos matrices con suma de sesgo.
// Calcula C = A × B + bias, donde A es la entrada (batch de features),
// B son los pesos de la conexión y bias el término independiente.
// Implementa la capa densa o fully-connected: cada neurona de salida recibe
// una combinación lineal de todas las neuronas de entrada. Es la capa utilizada
// en la etapa final de clasificación para proyectar las características extraídas
// por las convoluciones hacia el espacio de decisión de las clases.
// ============================================================
  __global__ void matmul_kernel(
      const float *A, const float *B, const float *bias,
      float *C, int M, int K, int N
  ) {
      int row = blockIdx.y * blockDim.y + threadIdx.y;
      int col = blockIdx.x * blockDim.x + threadIdx.x;
      if (row >= M || col >= N) return;
      float sum = 0.0f;
      for (int i = 0; i < K; i++) {
          sum += A[row * K + i] * B[i * N + col];
      }
      C[row * N + col] = sum + bias[col];
  }
// ============================================================
// batch_norm_kernel: Normaliza las activaciones usando estadísticas poblacionales.
// Recibe un vector de activaciones y lo normaliza con media y varianza aprendidas
// durante el entrenamiento, luego escala y desplaza con parámetros entrenables.
// Estabiliza la distribución de activaciones entre capas, permitiendo que la red
// converja más rápido y sea menos sensible a la inicialización de pesos.
// Se aplica después de las activaciones para mantener las magnitudes controladas.
// ============================================================
  __global__ void batch_norm_kernel(
      const float* input, const float* gamma, const float* beta,
      const float* mean, const float* var,
      float* output, int total_elements, int channels, float epsilon
  ) {
      int idx = blockIdx.x * blockDim.x + threadIdx.x;
      if (idx >= total_elements) return;
      int c = idx % channels;
      float x_hat = (input[idx] - mean[c]) / sqrtf(var[c] + epsilon);
      output[idx] = gamma[c] * x_hat + beta[c];
  }
// ============================================================
// softmax_kernel: Convierte un vector de puntuaciones en una distribución de probabilidad.
// Recibe los logits (puntuaciones crudas de cada clase) y devuelve probabilidades
// en el rango [0, 1] cuya suma es 1. La transformación es exponencial con sustracción
// del máximo para evitar desbordamiento numérico.
// Es la última capa del pipeline: transforma las salidas de la red en una distribución
// interpretable como la confianza del modelo en cada clase de Fashion MNIST.
// ============================================================
  __global__ void softmax_kernel(
      const float *input, float *output,
      int batch_size, int num_classes
  ) {
      int batch_idx = blockIdx.x;
      int tid = threadIdx.x;
      extern __shared__ float shared_mem[];
      if (batch_idx >= batch_size) return;
      int base_idx = batch_idx * num_classes;
      float max_val = -FLT_MAX;
      for (int i = tid; i < num_classes; i += blockDim.x) {
          max_val = fmax(max_val, input[base_idx + i]);
      }
      __shared__ float block_max;
      if (tid == 0) block_max = max_val;
      __syncthreads();
      for (int i = tid; i < num_classes; i += blockDim.x) {
          shared_mem[i] = expf(input[base_idx + i] - block_max);
      }
      __syncthreads();
      float sum = 0.0f;
      for (int i = tid; i < num_classes; i += blockDim.x) {
          sum += shared_mem[i];
      }
      __shared__ float block_sum;
      if (tid == 0) block_sum = 0.0f;
      __syncthreads();
      atomicAdd(&block_sum, sum);
      __syncthreads();
      for (int i = tid; i < num_classes; i += blockDim.x) {
          output[base_idx + i] = shared_mem[i] / block_sum;
      }
  }
}
"""

with open("kernels.cu", "w") as f:
    f.write(CUDA_KERNELS_SOURCE)

import os
cmd = (
    "nvcc --fatbin "
    "-gencode arch=compute_61,code=sm_61 "
    "-gencode arch=compute_75,code=sm_75 "
    "-gencode arch=compute_86,code=sm_86 "
    "-gencode arch=compute_89,code=sm_89 "
    "kernels.cu -o kernels.fatbin"
)
os.system(cmd)
print("Archivo kernels.fatbin generado exitosamente!")